### スパースモデリングによるトモグラフ像の再構成

In [ ]:
import pandas as pd
from sklearn.metrics import mean_squared_error
import numpy as np
from scipy import sparse
#from scipy import ndimage
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
import os
%matplotlib inline


In [ ]:
import os
os.makedirs("image_executed", exist_ok=True)


**projection operatorの生成**

以下ではoriginalのコードの名前をそのまま用いているので対応がわかりにくいですが

- proj_operator = X
- data = w
- proj = y 

という対応関係です。

In [ ]:
# scikit learnのコード、文献[3]

def _weights(x, dx=1, orig=0):
    x = np.ravel(x)
    floor_x = np.floor((x - orig) / dx).astype(np.int64)
    alpha = (x - orig - floor_x * dx) / dx
    return np.hstack((floor_x, floor_x + 1)), np.hstack((1 - alpha, alpha))


def _generate_center_coordinates(l_x):
    X, Y = np.mgrid[:l_x, :l_x].astype(np.float64)
    center = l_x / 2.
    X += 0.5 - center
    Y += 0.5 - center
    return X, Y


def build_projection_operator(l_x, n_dir):
    """ Compute the tomography design matrix.

    Parameters
    ----------

    l_x : int
        linear size of image array

    n_dir : int
        number of angles at which projections are acquired.

    Returns
    -------
    p : sparse matrix of shape (n_dir l_x, l_x**2)
    """
    X, Y = _generate_center_coordinates(l_x)
    angles = np.linspace(0, np.pi, n_dir, endpoint=False)
    data_inds, weights, camera_inds = [], [], []
    data_unravel_indices = np.arange(l_x ** 2)
    data_unravel_indices = np.hstack((data_unravel_indices,
                                      data_unravel_indices))
    for i, angle in enumerate(angles):
        Xrot = np.cos(angle) * X - np.sin(angle) * Y

        inds, w = _weights(Xrot, dx=1, orig=X.min())
        mask = np.logical_and(inds >= 0, inds < l_x)

        weights += list(w[mask])
        camera_inds += list(inds[mask] + i * l_x)
        data_inds += list(data_unravel_indices[mask])
    proj_operator = sparse.coo_matrix((weights, (camera_inds, data_inds)))
    return proj_operator


**測定画像の読み込み**

トモグラフィ像にnoiseを加えて観測像を目的変数$\vec y$とする。

In [ ]:
def get_data(prefix = "../data/font"):
    """load image data.

    Returns:
        [np.array]: image data.
    """
    # data directory is prefix.

    # "dance64.csv"
    # "syou64.csv"
    # "zen64.csv"
    filename = os.path.join(prefix, "L64.csv")

    data = np.loadtxt(filename, delimiter=",")
    return data


g_data = get_data()


データの規格化をして可視化する。


In [ ]:
def normalize_data(data):
    """画像値の規格化

    Args:
        data (np.array): image

    Returns:
        np.array: 規格化された画像
    """
    m1 = data.ravel().min()
    m2 = data.ravel().max()
    """
    normalize data
    """
    data = (data-m1)/(m2-m1)
    return data
# the range of data is now [0:1]


g_w = normalize_data(g_data)
print(g_w.shape)



data行列を白黒画像として表示


In [ ]:
plt.imshow(g_data, cmap=plt.cm.gray, interpolation='nearest')


#### wをhistgram表示する。多数が０であることが分かる。

In [ ]:
plt.hist(g_data.ravel(), bins=100)


パラメタの設定を行う。
以下の設定ではNの1/dのデータを用いて再構成を行ってう。
alpha1, alpha2は予め決めておいたLasso,Ridge回帰のハイパーパラメタである。

In [ ]:
g_d = 4
g_noise_fac = 0.05
g_alpha1 = 10**(-5)
g_alpha2 = 1
g_fit_intercept = True


平行系tomographyのシミュレーション

$$
= X w + noise\_fac \times N(0,1) 
$$

を行う。


まず回転行列$X$の生成を行う。

In [ ]:
def make_X(l: int, d: int):
    N = l*l
    print("N=", l*l)
    print("P=", l*l//d)
    print("P/N= {}%".format((l//d)/l*100))
    """
    construcut a tomography image
    l//d = int(l/d) = the number of angles 
    """
    X = build_projection_operator(l, l//d)
    print("X.shape=", X.shape)
    return X


g_X = make_X(g_w.shape[0], g_d)


ここで用いているXは特殊なmatrix型です。

In [ ]:
type(g_X)


ノイズを含めて$y$の生成を行う。

In [ ]:
def make_y(X, w, noise_fac):
    y = X * w.reshape(-1)[:, np.newaxis]
    print("add {}*N(0,1) to y".format(noise_fac))
    y += + noise_fac * np.random.randn(*y.shape)
    y = y.reshape(-1)
    print("y.shape=", y.shape)
    return y


g_y = make_y(g_X, g_w, g_noise_fac)


次のコードはセーブするために用いる。（このscriptでは用いないが保存しておく。）

In [ ]:
import os


def save_Xy_file(X, y, w, d, prefix="data_executed", postfix=None):
    if not os.path.isdir(prefix):
        os.makedirs(prefix)
    df = pd.DataFrame(w.ravel()[:, np.newaxis])
    filename = os.path.join(prefix, "w_{}.csv".format(postfix))
    df.to_csv(filename, index=False, float_format="%.2f", header=False)
    print("file saved to", filename)
    df1 = pd.DataFrame(X.toarray())
    df2 = pd.DataFrame(y)
    df = pd.concat([df1, df2], axis=1)
    filename = os.path.join(prefix, "Xy-{}_{}.csv".format(str(d), postfix))
    df.to_csv(filename, index=False,
              float_format="%.2f", header=False)
    print("file saved to", filename)


save_Xy_file(g_X, g_y, g_w, g_d, postfix="L64")


トモグラフィシミュレーション結果から画像の再構成を行う。

Lassoにより$w$を求める。
最後のrmseは係数の間の一致具合を見ている。

In [ ]:
def L1_reconstruction(X, y, alpha1, fit_intercept, l):
    """Reconstruction with L1 (Lasso) penalization

    y = X w
    の係数wを求める。
    X = proj_operator
    y = proj
    w = rec_l1

    Args:
        X (np.array): descriptor
        y (np.array): target values
        alpha1 (float): hyperparameter of Ridge regression
        fit_intercept (bool): fit intercept or not

    Returns:
        np.array: coefficients of linear function
    """
    rgr_lasso = Lasso(alpha=alpha1, fit_intercept=fit_intercept)

    rgr_lasso.fit(X, y)
    w_l1 = rgr_lasso.coef_.reshape(l, l)
    if fit_intercept:
        print("LASSO intercept", rgr_lasso.intercept_)
    return w_l1


g_w_l1 = L1_reconstruction(g_X, g_y, g_alpha1, g_fit_intercept, g_w.shape[0])
print(g_w_l1.reshape(-1).shape)
g_rmse1 = np.sqrt(mean_squared_error(g_w_l1, g_w))
print("rmse(L1)=", g_rmse1)


Ridge回帰によりwを求める。

In [ ]:
def L2_reconstruction(X, y, alpha2, fit_intercept, l):
    """Reconstruction with L2 (Ridge) penalization

    y = X w
    の係数wを求める。
    X = proj_operator
    y = proj
    w = rec_l1

    Args:
        X (np.array): descriptor
        y (np.array): target values
        alpha1 (float): hyperparameter of Ridge regression
        fit_intercept (bool): fit intercept or not

    Returns:
        np.array: coefficients of linear function
    """
    rgr_ridge = Ridge(alpha=alpha2, fit_intercept=fit_intercept)

    rgr_ridge.fit(X, y)
    w_l2 = rgr_ridge.coef_.reshape(l, l)
    if fit_intercept:
        print("ridge intercept", rgr_ridge.intercept_)
    return w_l2


g_w_l2 = L2_reconstruction(g_X, g_y, g_alpha2, g_fit_intercept, g_w.shape[0])
g_rmse2 = np.sqrt(mean_squared_error(g_w_l2, g_w))
print("rmse(L2)=", g_rmse2)


In [ ]:
def show_images(w, w2, w_title, w2_title, filename=None):
    """再構成した図をならべて表示する。

    Args:
        w (np.array): image 1
        w2 (np.array): image 2
        w_title (str): title 1
        w2_title (str): title 2
        filename (str, optional): filename. Defaults to None.
    """
    plt.figure(figsize=(8, 3.3))
    plt.subplot(121)
    plt.imshow(w, cmap=plt.cm.gray, interpolation='nearest')
    plt.title(w_title)
    plt.axis('off')
    plt.subplot(122)
    plt.imshow(w2, cmap=plt.cm.gray, interpolation='nearest')
    plt.title(w2_title)
    plt.axis('off')
    if filename is not None:
        plt.savefig(filename)
    plt.show()

    plt.figure(figsize=(8, 3.3))
    plt.subplot(121)
    plt.hist(w.reshape(-1), bins=100)
    plt.title(w_title)
    plt.subplot(122)
    plt.hist(w2.reshape(-1), bins=100)
    plt.title(w2_title)

    plt.show()


show_images(g_w, g_w_l1, "original image", "L1 reconstructed",
            filename="image_executed/letter_l1_reconstructed.png")


In [ ]:
show_images(g_w, g_w_l2, "original image", "L2 reconstructed")


In [ ]:
# 0未満を0に、１以上を１に変換する。
# （論文の図示でそれを言わずに行うと捏造となる。）

g_w_l1_modified = g_w_l1.copy()
g_w_l1_modified[g_w_l1_modified>1]=1.0
g_w_l1_modified[g_w_l1_modified<0]=0.0

show_images(g_w, g_w_l1_modified, "original image", "L1 reconstructed",)

**問題**

data directoryには他にもフォントがあります。画像再構成を試してみてください。

1. L64.csv  
2. dance64.csv  
3. syou64.csv 
4. zen64.csv


**参考文献**

L以外のフォントは以下のurlから取得しました。

1. 日本語フォント「筆文字フリー素材集」　http://fudemoji-free.com/


scikit learnの original codeb部分は

2. author: Emmanuelle Gouillart <emmanuelle.gouillart@nsup.org>
